First download the dataset from: https://www.kaggle.com/code/koshirosato/shutterstock-dataset-for-ai-vs-human-gen-image

Extract it into where you want to work

The link also includes a sample notebook. Some code is based off of it.

In [6]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [8]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.10.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Loading Dataset

In [5]:
curr_dir = os.getcwd()
#print(curr_dir)

## Creating Dataset (Only run if data.csv does not exist)

In [13]:
train_csv = pd.read_csv(curr_dir + "\\train.csv")
test_csv = pd.read_csv(curr_dir + "\\test_v2.csv")
test_labels_csv = pd.read_csv(curr_dir + "\\test_v2_labels.csv")

print(f'Train shape: {train_csv.shape}')
print(f'Test shape: {test_csv.shape}')
print(f'Test labels shape: {test_labels_csv}')

Train shape: (79950, 3)
Test shape: (5540, 1)
Test labels shape:                                                      id  label
0     test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg    1.0
1     test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg    1.0
2     test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg    0.0
3     test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg    0.0
4     test_data_v2/a16495c578b7494683805484ca27cf9f.jpg    0.0
...                                                 ...    ...
5535  test_data_v2/483412064ff74d9d9472d606b65976d9.jpg    1.0
5536  test_data_v2/c0b49ba4081a4197b422dac7c15aea7f.jpg    0.0
5537  test_data_v2/01454aaedec140c0a3ca1f48028c41cf.jpg    0.0
5538  test_data_v2/e9adfea8b67e4791968c4c2bdd8ec343.jpg    1.0
5539  test_data_v2/ba8f4198e8d74d3394fa56c56af23442.jpg    1.0

[5540 rows x 2 columns]


These are CSV files with the filepath + label, will need to use the actual image files for training. Also the image files are not of the same size, though for all of the images one of the dimensions is 768.

The current Train/Test split is 79950 to 5540. I am going to reformat the set so that it forms an 80/20 split for Train/Test. (Validation set will be split from Training set and it will also be 80/20). Final Data Distribution: 64% training, 16% validation, 20% test

In [14]:
test_labels_csv = test_labels_csv[['id', 'label']].rename(columns={'id': 'file_name'})
test_labels_csv['label'] = test_labels_csv['label'].astype('int64')
print(test_labels_csv.head(5))
train_csv = train_csv[['file_name', 'label']]
print(train_csv.head(5))

                                           file_name  label
0  test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg      1
1  test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg      1
2  test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg      0
3  test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg      0
4  test_data_v2/a16495c578b7494683805484ca27cf9f.jpg      0
                                         file_name  label
0  train_data/a6dcb93f596a43249135678dfcfc17ea.jpg      1
1  train_data/041be3153810433ab146bc97d5af505c.jpg      0
2  train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg      1
3  train_data/8542fe161d9147be8e835e50c0de39cd.jpg      0
4  train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg      1


In [18]:
data = pd.concat([train_csv, test_labels_csv], ignore_index = True)
print(data.sample(frac = 1).head(5))

                                               file_name  label
49631    train_data/65b77453cdc044fd99ac25b189d09f15.jpg      0
28971    train_data/29a2421d72fe43e3a1265582f45d53bb.jpg      0
53085    train_data/aec72301d87746d1b18c93f26125a173.jpg      0
84076  test_data_v2/dbab047ee39f4025a7cd90aa66b40a71.jpg      1
41638    train_data/70e3bb4dcef447239084bfce6a8fc869.jpg      1


## Loading Data

In [6]:
file_name = curr_dir + "\\data.csv"
if not os.path.isfile(file_name):
    data.to_csv(file_name)
else:
    data = pd.read_csv(file_name)

In [7]:
SEED = 694201337

Creating Train, Validation, Test Datasets (note still need to load the images later)

In [8]:
train_df, test_df = train_test_split(data, 
                                     test_size=0.2, 
                                     random_state=SEED, 
                                     stratify=data['label'])
train_df, val_df = train_test_split(train_df,
                                    test_size=0.2,
                                    random_state=SEED,
                                    stratify=train_df['label'])

In [9]:
display(train_df['label'].value_counts())
display(val_df['label'].value_counts())
display(test_df['label'].value_counts())

label
1    27428
0    27285
Name: count, dtype: int64

label
1    6858
0    6821
Name: count, dtype: int64

label
1    8571
0    8527
Name: count, dtype: int64

Now need to convert from file_name in data frames to the actual image. Need CV2 for this

Image size parameters.

In [2]:
IMG_WIDTH = 224
IMG_HEIGHT = 224

Function for resizing image with padding. (I think this will mess up training)

In [3]:
def resize_and_pad(img, target_size=768):
    h, w = img.shape[:2]

    # Scale so the larger dimension matches target_size
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    img_resized = cv2.resize(img, (new_w, new_h))

    # Pad to make it square (target_size x target_size)
    pad_top    = (target_size - new_h) // 2
    pad_bottom = target_size - new_h - pad_top
    pad_left   = (target_size - new_w) // 2
    pad_right  = target_size - new_w - pad_left

    img_padded = cv2.copyMakeBorder(
        img_resized, pad_top, pad_bottom, pad_left, pad_right,
        borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0)  # Black padding
    )
    return img_padded  # Shape: (768, 768, 3)

In [4]:
def create_dataset(df):
    imgs = []
    
    for path in df['file_name']:
        img = cv2.imread(curr_dir + "\\" + path)
        #img = resize_and_pad(img)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        imgs.append(img)
    imgs = np.array(imgs, dtype='float32')
    df = pd.get_dummies(df['label'])
    return imgs, df

In [ ]:
train_imgs, train_df = create_dataset(train_df)
val_imgs, val_df = create_dataset(val_df)
test_imgs, test_df = create_dataset(test_df)

train_imgs = train_imgs / 255
val_imgs = val_imgs / 255
test_imgs = test_imgs / 255

In [9]:
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)

model = Sequential()
model.add(tf.keras.layers.Input(input_shape))
model.add(tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D(3))
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Conv2D(32, kernel_size=3, padding='same', activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D(3))
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Conv2D(16, kernel_size=3, padding='same', activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D(3))
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(2, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 224, 224, 64)      1792      
                                                                 
 batch_normalization (BatchN  (None, 224, 224, 64)     256       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 64)       0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 74, 74, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 74, 74, 32)        18464     
                                                                 
 batch_normalization_1 (Batc  (None, 74, 74, 32)      

In [ ]:
import warnings

warnings.simplefilter('ignore')

es_callback = tf.keras.callbacks.EarlyStopping(patience=20,
                                               verbose=1,
                                               restore_best_weights=True)

history = model.fit(train_imgs,
                    train_df,
                    batch_size=256,
                    epochs=100,
                    steps_per_epoch=len(train_imgs)//256,
                    callbacks=es_callback,
                    validation_data=(val_imgs, val_df),
                    verbose=True)

In [ ]:
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot()
pd.DataFrame(history.history)[['loss', 'val_loss']].plot()

In [ ]:
model.evaluate(test_imgs, test_df)

In [ ]:
model.save("SampleModel.keras")